# M27 - CUDA intermediate MoE tensor parity (RORET off vs on)

Goal: on a Kaggle 2xT4 GPU runner, measure ENGINE-LEVEL raw intermediate MoE
tensor byte parity between RORET off and RORET on, for cpu_share 0.0 and 1.0.

Scope limits:
- Synthetic fixed-seed fixture only. No real weights.
- NO SPEED CLAIM. No timing or throughput measurement of any kind.
- PASS requires 0 differing bytes in the dumped raw tensor files.
- The measurement uses the checked-in harness tools/roret/check_intermediate_parity.py.
  This notebook only adds -ngl 99 and -t 1; it does not alter the harness gate.

Kernel logs are read (captured by run_logged and printed with tail), not sent to
an output API. M27_SUMMARY.json is written under /kaggle/working on every path.


In [ ]:
import os, sys, json, re, hashlib, subprocess, shutil

WORK = "/kaggle/working"
REPO = WORK + "/inkengine-llama"
GGUF = WORK + "/synthetic.gguf"


def run_logged(cmd, log_path, cwd=None, env=None, timeout=None):
    # timeout=None is required for the CUDA build, which can take many minutes.
    try:
        with open(log_path, "w") as f:
            p = subprocess.run(cmd, cwd=cwd, env=env, stdout=f,
                               stderr=subprocess.STDOUT, timeout=timeout)
        return p.returncode
    except subprocess.TimeoutExpired:
        with open(log_path, "a") as f:
            f.write("\nTIMEOUT: command exceeded " + str(timeout) + " seconds\n")
        return 124


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def tail(path, n=40):
    try:
        lines = open(path, errors="replace").read().splitlines()
    except OSError:
        return ""
    return "\n".join(lines[-n:])


summary = {"overall_verdict": "NO_GO", "blocker": "", "gate": {}}
PIPELINE_OK = True
print("python", sys.version.split()[0])


In [ ]:
if not PIPELINE_OK:
    print("SKIP: pipeline blocked")
else:
    if not os.path.isdir(REPO):
        cmd = ["git", "clone", "-b", "main",
               "https://github.com/10xdev4u-alt/inkengine-llama.git", REPO]
        print("RUN", " ".join(cmd))
        rc = subprocess.run(cmd).returncode
        print("clone exit", rc)
        if rc != 0:
            PIPELINE_OK = False
            summary["blocker"] = "git clone failed"
    else:
        print("clone: existing dir, skip", REPO)
    if PIPELINE_OK:
        r = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO,
                           capture_output=True, text=True)
        summary["head_sha"] = r.stdout.strip()
        print("HEAD", summary["head_sha"])
        if not summary["head_sha"]:
            PIPELINE_OK = False
            summary["blocker"] = "git rev-parse HEAD failed"


In [ ]:
if not PIPELINE_OK:
    print("SKIP: pipeline blocked")
else:
    cfg = ["cmake", "-S", ".", "-B", "build", "-DGGML_CUDA=ON",
           "-DCMAKE_CUDA_ARCHITECTURES=75", "-DLLAMA_BUILD_TESTS=OFF",
           "-DLLAMA_CURL=OFF", "-DGGML_CUDA_NO_VMM=ON"]
    summary["configure_command"] = cfg
    print("RUN", " ".join(cfg))
    rc1 = run_logged(cfg, WORK + "/cmake_configure.log", cwd=REPO)
    summary["configure_exit"] = rc1
    summary["configure_log_tail"] = tail(WORK + "/cmake_configure.log", n=20)
    print("configure exit", rc1)
    print("configure log tail:")
    print(summary["configure_log_tail"])

    bld = ["cmake", "--build", "build", "--target", "llama-debug", "-j4"]
    print("RUN", " ".join(bld))
    rc2 = run_logged(bld, WORK + "/cmake_build.log", cwd=REPO)
    summary["build_exit"] = rc2
    summary["build_log_tail"] = tail(WORK + "/cmake_build.log", n=20)
    print("build exit", rc2)
    print("build log tail:")
    print(summary["build_log_tail"])

    summary["build_ok"] = (rc1 == 0 and rc2 == 0)
    bin_path = REPO + "/build/bin/llama-debug"
    if summary["build_ok"]:
        if not (os.path.isfile(bin_path) and os.access(bin_path, os.X_OK)):
            summary["build_ok"] = False
            summary["blocker"] = "missing or non-executable build/bin/llama-debug"
            print("BLOCKER: missing or non-executable build/bin/llama-debug")
    PIPELINE_OK = summary["build_ok"]
    print("build_ok", summary["build_ok"])
    if not PIPELINE_OK:
        print("BLOCKER: CUDA build failed; later stages skipped")


In [ ]:
if not PIPELINE_OK:
    print("SKIP: pipeline blocked")
else:
    gen = REPO + "/tools/audit/make_synthetic_gguf.py"
    gen_env = dict(os.environ)
    gen_env["PYTHONPATH"] = REPO + "/gguf-py"
    cmd = [sys.executable, gen, "--out", GGUF]
    print("RUN", " ".join(cmd))
    rc = run_logged(cmd, WORK + "/make_gguf.log", cwd=REPO, env=gen_env)
    print("generator exit", rc)
    if rc != 0:
        print(tail(WORK + "/make_gguf.log", n=40))
    if rc == 0 and os.path.isfile(GGUF) and os.path.getsize(GGUF) > 0:
        summary["fixture_sha256"] = sha256_file(GGUF)
        print("fixture_sha256", summary["fixture_sha256"])
    else:
        summary["blocker"] = "GGUF generation failed or file missing/empty"
        PIPELINE_OK = False
        print("BLOCKER: GGUF generation failed or file missing/empty")


In [ ]:
if not PIPELINE_OK:
    print("SKIP: pipeline blocked")
else:
    summary["shares"] = {}
    harness = REPO + "/tools/roret/check_intermediate_parity.py"
    for share in ("0.0", "1.0"):
        out = WORK + "/m27-" + share
        cmd = [sys.executable, harness,
               "--llama-debug", REPO + "/build/bin/llama-debug",
               "--model", GGUF,
               "--out", out,
               "--cpu-share", share,
               "--ngl", "99",
               "--threads", "1"]
        print("RUN", " ".join(cmd))
        hlog = WORK + "/m27-" + share + "-harness.log"
        rc = run_logged(cmd, hlog, timeout=1800)
        rec = {
            "harness_exit": rc,
            "harness_stdout_tail": tail(hlog, n=60),
            "overall_verdict": "NO_GO",
            "off_exit": -1,
            "on_exit": -1,
            "off_hook": False,
            "on_hook": False,
            "tensors": 0,
            "bytes_compared": 0,
            "differing_bytes": 0,
        }
        spath = out + "/M26_SUMMARY.json"
        try:
            with open(spath) as f:
                hsum = json.load(f)
        except (OSError, ValueError) as exc:
            hsum = {}
            rec["summary_read_error"] = type(exc).__name__ + ": " + str(exc)
        rec["overall_verdict"] = hsum.get("overall_verdict", "NO_GO")
        rows = hsum.get("shares") or []
        hit = None
        for row in rows:
            if str(row.get("cpu_share")) == share:
                hit = row
                break
        if hit is None and len(rows) == 1:
            hit = rows[0]
        if hit is not None:
            for key in ("off_exit", "on_exit", "off_hook", "on_hook",
                        "tensors", "bytes_compared", "differing_bytes"):
                rec[key] = hit.get(key, rec[key])
        summary["shares"][share] = rec
        print("share", share, "harness_exit", rc, "verdict", rec["overall_verdict"])
        print("harness stdout tail (" + share + "):")
        print(rec["harness_stdout_tail"])


In [ ]:
expected_sha = "cf72921a76518bff78f969227be8b036a9dc869ca8c0eb5e78a66c89a81e9617"
if not PIPELINE_OK:
    summary["gate"]["pipeline_ok"] = False
else:
    gates = {
        "build_ok": summary.get("build_ok") is True,
        "fixture_sha256_match": summary.get("fixture_sha256") == expected_sha,
    }
    for share in ("0.0", "1.0"):
        rec = summary.get("shares", {}).get(share, {})
        gates["share_" + share + "_harness_exit_zero"] = rec.get("harness_exit") == 0
        gates["share_" + share + "_verdict_pass"] = rec.get("overall_verdict") == "PASS"
        gates["share_" + share + "_off_exit_zero"] = rec.get("off_exit") == 0
        gates["share_" + share + "_on_exit_zero"] = rec.get("on_exit") == 0
        gates["share_" + share + "_off_hook_false"] = rec.get("off_hook") is False
        gates["share_" + share + "_on_hook_true"] = rec.get("on_hook") is True
        gates["share_" + share + "_tensors_positive"] = (
            isinstance(rec.get("tensors"), int) and rec["tensors"] > 0)
        gates["share_" + share + "_bytes_positive"] = (
            isinstance(rec.get("bytes_compared"), int) and rec["bytes_compared"] > 0)
        gates["share_" + share + "_diff_zero"] = rec.get("differing_bytes") == 0
    summary["gate"] = gates
    failed = [k for k, v in gates.items() if not v]
    summary["blocker"] = "gate failed: " + ",".join(failed) if failed else ""

gate_ok = bool(summary["gate"]) and all(summary["gate"].values())
summary["overall_verdict"] = "PASS" if (PIPELINE_OK and gate_ok) else "NO_GO"
print("GATE:", json.dumps(summary["gate"], sort_keys=True))
print("OVERALL (gate):", summary["overall_verdict"])
if summary["blocker"]:
    print("BLOCKER:", summary["blocker"])


In [ ]:
out_path = WORK + "/M27_SUMMARY.json"
with open(out_path, "w") as f:
    json.dump(summary, f, indent=2, sort_keys=True)

print("OVERALL: " + summary["overall_verdict"])
print("BLOCKER: " + summary["blocker"])
for share in ("0.0", "1.0"):
    rec = summary.get("shares", {}).get(share) or {}
    print("share=%s harness_exit=%s verdict=%s off_exit=%s on_exit=%s off_hook=%s on_hook=%s tensors=%s bytes=%s diff=%s" % (
        share, rec.get("harness_exit"), rec.get("overall_verdict"),
        rec.get("off_exit"), rec.get("on_exit"), rec.get("off_hook"),
        rec.get("on_hook"), rec.get("tensors"),
        rec.get("bytes_compared"), rec.get("differing_bytes")))
    print("harness stdout tail (" + share + "):")
    print(rec.get("harness_stdout_tail", ""))
print("SUMMARY: " + out_path)
